In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

from pixell import enmap, enplot, curvedsky, wcsutils
import healpy as hp
from mnms import utils, tiled_noise

# tubes = [
#     'lat_f030',
#     'lat_f040',
#     'lat_f090',
#     'lat_f150',
#     'lat_f220',
#     'lat_f280'
# ]

tubes = [
    'lat_f090',
    'lat_f150',
    'lat_f220',
    'lat_f280'
]

In [ ]:
# first prep masks

# load dr4 mask and geometries. note dr4 mask in cc quad so need to shift pixels
# up by 0.5 (this is ok since this is all mock anyway)

ivar_dr4 = enmap.read_map('/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/act_dr4/maps/s15/act_dr4.01_s15_D56_pa3_f090_nohwp_night_3pass_4way_coadd_ivar.fits')
ivar_dr4.wcs.wcs.crpix[1] += 0.5 # wcs-y is second element
dr4_footprint = (ivar_dr4 != 0)

tube = 'lat_f090' # good for f090 and up
shape, wcs = enmap.read_map_geometry(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/footprint/so_geometry_deep56_v20250306_{tube}.fits')
mask_obs = enmap.zeros(shape, wcs, dtype=bool)
enmap.insert(mask_obs, dr4_footprint)
enmap.write_map('/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/masks/mask_obs_mf_uhf.fits', mask_obs.astype(np.int8))


In [ ]:
# now prep ivar

# we need to adjust the levels for each tube to match the forecast
sigma_dr4 = enmap.zeros(shape, wcs, dtype=np.float32) 
enmap.insert(sigma_dr4, ivar_dr4)
sigma_dr4 = np.divide(1, sigma_dr4, where=mask_obs, out=enmap.zeros(*mask_obs.geometry))**0.5
depth_dr4 = sigma_dr4 * sigma_dr4.pixsizemap()**0.5 * 180 * 60 / np.pi
median_depth_dr4 = np.median(depth_dr4[mask_obs])
print(f'median depth dr4: {median_depth_dr4:0.3f} uK-arcmin')

target_depth_f090 = 12 # uK-arcmin for ISO
target_depth_per_f090_tube = target_depth_f090 * 4**0.5 # 4 iso mf tubes
num_splits = 4

aso_depth_by_band = {
    'lat_f030': 61,
    'lat_f040': 30,
    'lat_f090': 5.3,
    'lat_f150': 6.6,
    'lat_f220': 15,
    'lat_f280': 35
}

aso_tubes_per_band = {
    'lat_f030': 1,
    'lat_f040': 1,
    'lat_f090': 8,
    'lat_f150': 8,
    'lat_f220': 4,
    'lat_f280': 4    
}

rel_depth_per_tube = {k: aso_depth_by_band[k] * aso_tubes_per_band[k]**0.5 for k in tubes}
rel_depth_per_tube = {k: rel_depth_per_tube[k] / rel_depth_per_tube['lat_f090'] for k in tubes}
iso_depth_per_split = {k: target_depth_per_f090_tube * rel_depth_per_tube[k] * num_splits**0.5 for k in tubes}
print('iso depth per split:', iso_depth_per_split)

for k in tubes:
    sigma_per_split = iso_depth_per_split[k] / median_depth_dr4 * sigma_dr4
    sigma_per_split /= (sigma_dr4.pixsizemap()**0.5 * 180 * 60 / np.pi)
    ivar_per_split = np.divide(1, sigma_per_split**2, where=mask_obs, out=enmap.zeros(*mask_obs.geometry, dtype=np.float32))
    enmap.write_map(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/maps/ivar_{k}_4way_setX.fits', ivar_per_split)
    enmap.write_map(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/maps/ivar_{k}_4way_coadd.fits', ivar_per_split * num_splits)

In [ ]:
# prep noise models

# the cov ells in the act noise models are after appropriate
# sqrt_ivar flattening (eg, so dg4 cov ell is exactly 4 times dg2). so 
# all of the depth stuff is captured in the ivars we just made. we do need to 
# make two changes:

# first, we need to rescale the cov ells a little bit so they match onto the
# ivar level at high ell. we'll do the correction using the last value of the 
# dg2 TT spectra. because of the pixwin tilt, the correction is OK for 
# intermediate-ell pol spectra, but it's a little high for high-ell. But this is
# OK because we will smooth the ivar realization so there isn't a kink.

sqrt_cov_mat, _, extra_datasets_dict = tiled_noise.read_tiled_ndmap('/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/dr6v4/simulations/noise/models/act_dr6v4_tile_cmbmask_pa5_f090_pa5_f150_lmax10800_4way_set0_noise_model.hdf5',
                                                                    extra_datasets=['sqrt_cov_ell'])
scedg2 = extra_datasets_dict['sqrt_cov_ell']
cedg2 = scedg2.reshape(6, 6, -1)
cedg2 = utils.eigpow(cedg2, 2, axes=[0, 1])

# the correction is independent of the absolute ivar level because the cov ells
# are flattened, so we just assume std normal noise (this is true for the 
# noise we are trying to match onto, which will be std normal after ivar flatt-
# ening)
white_cedg = np.sum(1**2*enmap.pixsizemap(shape, wcs) * enmap.pixsizemap(shape, wcs))/(4*np.pi) # white noise pseudospectrum in patch
white_cedg /= np.sum(1**2 * enmap.pixsizemap(shape, wcs))/(4*np.pi) # w2 of patch
fac = np.zeros((6, 6, 1))

fac[0, 0] = white_cedg / cedg2[0, 0, -1]
fac[1, 1] = fac[0, 0]
fac[2, 2] = fac[0, 0]
fac[3, 3] = white_cedg / cedg2[3, 3, -1]
fac[4, 4] = fac[3, 3]
fac[5, 5] = fac[3, 3]

for i in range(6):
    for j in range(6):
        if i == j:
            continue
        fac[i, j] = np.sqrt(fac[i, i] * fac[j, j]) # keep the corr constant

print(fac.squeeze())
cedg2 *= fac

# next, we want to shift the power spectra left and right depending on the band
# of the tube (shift right for uhf, left for lf). we do this by inserting or
# deleting a constant value at l=2 and l=lmax, and defining
# the amount of shift just based on a gut sense. note we assume bands pair-off
# per tube like dr6, so we just list this by the first band in the tube

shift_by_band = {
    'lat_f030': -300,
    'lat_f090': 0,
    'lat_f220': 1000,
}

band_pairs = {
    'lat_f030': 'lat_f040',
    'lat_f090': 'lat_f150',
    'lat_f220': 'lat_f280'
}

scedg2s = {}
for k in tubes:
    if k in band_pairs:
        shift = shift_by_band[k]
        band_pair = (k, band_pairs[k])

        if shift < 0:
            this_cedg2 = np.zeros_like(cedg2)
            
            for i in range(6):
                for j in range(6):
                    seq_of_arrs = (cedg2[i, j, :2], cedg2[i, j, shift+2:], cedg2[i, j, -1]*np.ones(shift, dtype=cedg2.dtype))
                    this_cedg2[i, j] = np.concatenate(seq_of_arrs)
        
        elif shift > 0:
            this_cedg2 = np.zeros_like(cedg2)
            
            for i in range(6):
                for j in range(6):
                    seq_of_arrs = (cedg2[i, j, :2], cedg2[i, j, 2]*np.ones(shift, dtype=cedg2.dtype), cedg2[i, j, 2:-shift])
                    this_cedg2[i, j] = np.concatenate(seq_of_arrs)  

        else:
            this_cedg2 = cedg2

        this_scedg2 = utils.eigpow(this_cedg2, 0.5, axes=[0, 1])
        scedg2s[band_pair] = this_scedg2.reshape(2, 1, 3, 2, 1, 3, -1)

# now save the noise models
for band_pair in scedg2s:
    fn = '/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/simulations/noise/models/'
    tiled_noise.write_tiled_ndmap(fn + f"lat_piso_tile_cmbmask_{'_'.join(band_pair)}_lmax10800_4way_setX_noise_model.hdf5",
                                  sqrt_cov_mat,
                                  extra_datasets={'sqrt_cov_ell': scedg2s[band_pair]})

In [ ]:
# prep beams and leakage

# we use gaussian beams from the aso paper, to the lmax of the dr6 beams
# we use sqrt(3)x the dr6 error modes (3x at covariance level)
beam_dr6 = np.loadtxt('/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/dr6v4/beams/coadd_pa5_f090_night_beam_tform_jitter_cmb.txt')
rel_beam_dr6_error_modes = beam_dr6[:, 2:] / beam_dr6[:, 1][:, None] # normalize by b_l
rel_beam_dr6_error_modes = rel_beam_dr6_error_modes * np.sqrt(3)
l = np.arange(rel_beam_dr6_error_modes.shape[0])

fwhm_by_band = { # arcmin
    'lat_f030': 7.4,
    'lat_f040': 5.1,
    'lat_f090': 2.2,
    'lat_f150': 1.4,
    'lat_f220': 1.0,
    'lat_f280': 0.9    
}

for k in fwhm_by_band:
    bl = hp.gauss_beam(np.deg2rad(fwhm_by_band[k]/60), lmax=l[-1])
    beam = np.zeros((l.size, rel_beam_dr6_error_modes.shape[1]+2))
    beam[:, 0] = l
    beam[:, 1] = bl
    beam[:, 2:] = rel_beam_dr6_error_modes * bl[:, None]
    np.savetxt(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/beams/main_beams/beam_{k}.txt', beam)

# we use the same leakage as dr6
# we use sqrt(3)x the dr6 error modes (3x at the covariance level)
for p in ['e', 'b']:
    leakage = np.loadtxt(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/dr6v4/beams/pa5_f090_gamma_t2{p}.txt')
    leakage[:, 1:] *= np.sqrt(3)
    for k in fwhm_by_band:
        np.savetxt(f'/scratch/gpfs/SIMONSOBS/users/zatkins/projects/lat-iso/piso/misc/piso_sims/so/beams/leakage_beams/gamma_t2{p}_{k}.txt', leakage)